![Week 3 Day 3](../../assets/banners/w3d3.png)

# Week 3 Day 3 - Solution: Extending the Rubric Checker

The exercise: a new rule (every criterion needs a keyword), a repair for it, and a `second_marker` step for questions worth 5 marks or more.

⏮ Back to the lab: `week3/day3.ipynb`

In [ ]:
import copy
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from gradewise.config import DATA_DIR
from gradewise.schemas import ExamPaper

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in exam.questions}

In [ ]:
from gradewise.display import show_graph

class CheckState(TypedDict):
    question: dict
    problems: list[str]
    attempts: int
    max_attempts: int
    log: Annotated[list[str], operator.add]


def check(state: CheckState) -> dict:
    q = state["question"]
    total = round(sum(c["marks"] for c in q["criteria"]), 2)
    problems = []
    if total != q["max_marks"]:
        problems.append(f"criteria add up to {total:g}, but the question is worth {q['max_marks']:g}")
    if not q["methodology"].strip():
        problems.append("no marking methodology")
    for c in q["criteria"]:  # 💡 the new rule
        if not c["keywords"]:
            problems.append(f"{c['id']} has no keywords")
    return {"problems": problems, "log": [f"check   → {problems or 'clean ✅'}"]}


def longest_words(text: str, n: int = 2) -> list[str]:
    words = {w.strip(".,:;()").lower() for w in text.split()}
    return sorted(words, key=len, reverse=True)[:n]


def repair(state: CheckState) -> dict:
    q = copy.deepcopy(state["question"])
    problem = state["problems"][0]
    if problem.startswith("criteria"):
        q["criteria"][-1]["marks"] -= sum(c["marks"] for c in q["criteria"]) - q["max_marks"]
    elif problem == "no marking methodology":
        q["methodology"] = "Award each criterion independently. Ignore spelling."
    else:  # "<id> has no keywords"
        criterion = next(c for c in q["criteria"] if problem.startswith(c["id"] + " "))
        criterion["keywords"] = longest_words(criterion["description"])
    attempt = state["attempts"] + 1
    return {"question": q, "attempts": attempt, "log": [f"repair  → attempt {attempt}: fixed '{problem}'"]}


def second_marker(state: CheckState) -> dict:
    return {
        "log": [
            f"second  → 👥 {state['question']['id']} ({state['question']['max_marks']:g} marks) needs a second marker"
        ]
    }


def approve(state: CheckState) -> dict:
    return {"log": ["approve → ready for grading 🎉"]}


def escalate(state: CheckState) -> dict:
    return {"log": [f"escalate → Emma, please fix by hand: {state['problems']}"]}


def after_check(state: CheckState) -> Literal["second_marker", "approve", "repair", "escalate"]:
    if not state["problems"]:
        return "second_marker" if state["question"]["max_marks"] >= 5 else "approve"
    if state["attempts"] < state["max_attempts"]:
        return "repair"
    return "escalate"


builder = StateGraph(CheckState)
for fn in (check, repair, second_marker, approve, escalate):
    builder.add_node(fn.__name__, fn)
builder.add_edge(START, "check")
builder.add_conditional_edges("check", after_check, ["second_marker", "approve", "repair", "escalate"])
builder.add_edge("repair", "check")
builder.add_edge("second_marker", "approve")
builder.add_edge("approve", END)
builder.add_edge("escalate", END)
checker = builder.compile()
show_graph(checker)

In [ ]:
def broken(qid: str) -> dict:
    q = copy.deepcopy(QUESTIONS[qid].model_dump())
    q["criteria"][0]["marks"] += 1  # ⚠️ marks don't add up
    q["criteria"][1]["keywords"] = []  # ⚠️ a criterion with no keywords
    return q


for qid in ["Q5", "Q3"]:
    print(f"── {qid} ({QUESTIONS[qid].max_marks:g} marks) ──")
    result = checker.invoke({"question": broken(qid), "problems": [], "attempts": 0, "max_attempts": 4, "log": []})
    print("\n".join(result["log"]))
    assert ("second marker" in result["log"][-2]) == (QUESTIONS[qid].max_marks >= 5)